# Experiment 8 - RNN for Temporal Classification and Sequence-to-Sequence Modelling

**Aim:** Use recurrent neural networks for two sequence tasks:

1. **Temporal classification:** classify IMDB movie reviews as positive or
   negative with an LSTM.
2. **Sequence-to-sequence modelling:** train an encoder-decoder LSTM to
   reverse a short integer sequence.

**Theory in brief:** An RNN maintains a hidden state across time steps, so it
can model order and context. The LSTM variant uses input, forget, and output
gates to retain information over long spans and reduce vanishing gradients.
For sentiment classification, words are mapped to dense vectors by an
`Embedding` layer, padded to a fixed length, and passed through an LSTM to a
sigmoid output. For sequence-to-sequence modelling, the encoder LSTM compresses
the source sequence into a fixed vector, `RepeatVector` copies that vector
across the target time steps, and the decoder LSTM with a time-distributed
softmax emits one token per step.

**Datasets:**

- IMDB reviews: `https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb.npz`
- IMDB word index: `https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb_word_index.json`
- Sequence-reversal data: generated deterministically in code.

No local dataset is used.

**Expected result (original lab record):** IMDB sentiment accuracy about
**86%**; sequence-reversal token accuracy about **99.87%**.

**Runtime:** about 3-8 minutes on CPU for both parts; faster on GPU.

## 1. Environment setup

In [ ]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["PYTHONHASHSEED"] = "42"

import json
import random
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

## Part A - IMDB sentiment classification

### 2. Download IMDB and build the word index

`tf.keras.datasets.imdb.load_data` expects its cached file to live at the
default Keras cache location. The cell therefore downloads the explicit URLs
first and then calls the standard loader, which reuses the cached files and
applies the `num_words` vocabulary limit.

In [ ]:
IMDB_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb.npz"
IMDB_WORD_INDEX_URL = (
    "https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb_word_index.json"
)
VOCAB_SIZE = 10000
MAXLEN = 200

tf.keras.utils.get_file("imdb.npz", IMDB_URL, cache_subdir="datasets")
word_index_path = tf.keras.utils.get_file(
    "imdb_word_index.json",
    IMDB_WORD_INDEX_URL,
    cache_subdir="datasets",
)

(X_train_imdb, y_train_imdb), (X_test_imdb, y_test_imdb) = tf.keras.datasets.imdb.load_data(
    num_words=VOCAB_SIZE
)

X_train_imdb = tf.keras.utils.pad_sequences(
    X_train_imdb, maxlen=MAXLEN, padding="post", truncating="post"
)
X_test_imdb = tf.keras.utils.pad_sequences(
    X_test_imdb, maxlen=MAXLEN, padding="post", truncating="post"
)

with open(word_index_path, "r", encoding="utf-8") as file:
    raw_word_index = json.load(file)

word_index = {
    word: index + 3
    for word, index in raw_word_index.items()
    if index + 3 < VOCAB_SIZE
}
word_index["<PAD>"] = 0
word_index["<START>"] = 1
word_index["<UNK>"] = 2
word_index["<UNUSED>"] = 3
reverse_word_index = {index: word for word, index in word_index.items()}

print(f"Training sequences: {X_train_imdb.shape}")
print(f"Test sequences    : {X_test_imdb.shape}")
print(f"Vocabulary size   : {VOCAB_SIZE}")
print(f"Raw word index entries: {len(raw_word_index)}")
print(f"Training labels: {np.bincount(y_train_imdb).tolist()}")
print(f"Test labels    : {np.bincount(y_test_imdb).tolist()}")

assert X_train_imdb.shape[1] == MAXLEN
assert X_test_imdb.shape[1] == MAXLEN

In [ ]:
def decode_review(sequence):
    words = []
    for index in sequence:
        index = int(index)
        if index == 0:
            continue
        words.append(reverse_word_index.get(index, "?"))
    return " ".join(words)


def encode_review(text):
    tokens = re.findall(r"[a-z0-9']+", text.lower())
    indices = [1]
    for token in tokens:
        index = word_index.get(token, 2)
        indices.append(index if index < VOCAB_SIZE else 2)
    encoded = tf.keras.utils.pad_sequences(
        [indices],
        maxlen=MAXLEN,
        padding="post",
        truncating="post",
    )
    return encoded[0]


sample_index = 0
print("Label:", int(y_train_imdb[sample_index]))
print("Decoded review:")
print(decode_review(X_train_imdb[sample_index])[:1200])

### 3. LSTM sentiment classifier

The model uses a 10,000-word embedding table with 64-dimensional vectors, a
64-unit LSTM, and a sigmoid output. It is trained with Adam and binary
cross-entropy.

In [ ]:
def build_imdb_model(seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(MAXLEN,), name="review_tokens"),
            tf.keras.layers.Embedding(
                input_dim=VOCAB_SIZE,
                output_dim=64,
                name="word_embedding",
            ),
            tf.keras.layers.LSTM(64, name="review_lstm"),
            tf.keras.layers.Dense(1, activation="sigmoid", name="sentiment"),
        ],
        name="imdb_lstm_classifier",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


imdb_model = build_imdb_model()
imdb_model.summary()

In [ ]:
imdb_checkpoint = OUTPUT_DIR / "exp08_imdb_best.keras"
imdb_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(imdb_checkpoint),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=1,
        restore_best_weights=True,
        verbose=1,
    ),
]

start_time = time.perf_counter()
imdb_history = imdb_model.fit(
    X_train_imdb,
    y_train_imdb,
    epochs=3,
    batch_size=256,
    validation_split=0.1,
    verbose=2,
    callbacks=imdb_callbacks,
)
imdb_seconds = time.perf_counter() - start_time

imdb_test_loss, imdb_test_accuracy = imdb_model.evaluate(
    X_test_imdb, y_test_imdb, verbose=0
)
print(f"IMDB training time : {imdb_seconds:.2f} seconds")
print(f"IMDB test loss     : {imdb_test_loss:.4f}")
print(f"IMDB test accuracy : {imdb_test_accuracy:.4f}")

In [ ]:
imdb_probabilities = imdb_model.predict(X_test_imdb, verbose=0).ravel()
imdb_predictions = (imdb_probabilities >= 0.5).astype(int)

print("IMDB classification report:")
print(
    classification_report(
        y_test_imdb,
        imdb_predictions,
        target_names=["negative", "positive"],
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(y_test_imdb, imdb_predictions)
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(cm, cmap="Blues")
ax.set_title("IMDB sentiment confusion matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["negative", "positive"])
ax.set_yticklabels(["negative", "positive"])
for row in range(2):
    for column in range(2):
        color = "white" if cm[row, column] > cm.max() / 2 else "black"
        ax.text(column, row, str(cm[row, column]), ha="center", va="center", color=color)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp08_imdb_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(imdb_history.history["loss"], label="Train", linewidth=2)
axes[0].plot(imdb_history.history["val_loss"], label="Validation", linewidth=2)
axes[0].set_title("IMDB binary cross-entropy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(imdb_history.history["accuracy"], label="Train", linewidth=2)
axes[1].plot(imdb_history.history["val_accuracy"], label="Validation", linewidth=2)
axes[1].set_title("IMDB accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.0, 1.0)
axes[1].grid(alpha=0.3)
axes[1].legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp08_imdb_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 4. Sentiment inference on raw text

`encode_review` uses the same vocabulary and padding as training. It prepends
the start token, maps unknown words to the out-of-vocabulary index, and pads
or truncates to 200 tokens.

In [ ]:
def predict_sentiment(text):
    encoded = encode_review(text)
    probability = float(imdb_model.predict(encoded.reshape(1, MAXLEN), verbose=0)[0, 0])
    label = "positive" if probability >= 0.5 else "negative"
    return label, probability


example_reviews = [
    "This movie was absolutely wonderful, with brilliant acting and a moving story.",
    "A boring, terrible film. I regret watching it and would not recommend it.",
    "The plot was predictable but the performances were enjoyable.",
]

for review in example_reviews:
    label, probability = predict_sentiment(review)
    print(f"[{label:8s}] P(positive)={probability:.4f} | {review}")

## Part B - Sequence-to-sequence reversal

### 5. Generate the reversal dataset

Each source sequence is six integers in the range 1-11. The target is the same
sequence reversed. The model must learn to copy information through a fixed
encoder vector and emit it in reverse order.

In [ ]:
T = 6
V = 12
N_TRAIN_S2S = 8000
N_TEST_S2S = 1000

rng = np.random.default_rng(SEED)
src_train = rng.integers(1, V, size=(N_TRAIN_S2S, T), dtype=np.int32)
tgt_train = src_train[:, ::-1].copy()
src_test = rng.integers(1, V, size=(N_TEST_S2S, T), dtype=np.int32)
tgt_test = src_test[:, ::-1].copy()

print(f"Source train: {src_train.shape}, target train: {tgt_train.shape}")
print(f"Source test : {src_test.shape}, target test : {tgt_test.shape}")
print("Example source:", src_train[0].tolist())
print("Example target:", tgt_train[0].tolist())

assert np.array_equal(tgt_train[0], src_train[0][::-1])

### 6. Encoder-decoder LSTM

- Encoder: Embedding -> LSTM(128) -> fixed vector
- Bridge: `RepeatVector(T)` repeats the vector for each output step
- Decoder: LSTM(128, `return_sequences=True`) -> TimeDistributed Dense(V, softmax)

The loss is sparse categorical cross-entropy over the target token at each
time step.

In [ ]:
def build_seq2seq_model(seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(T,), dtype="int32", name="source_sequence"),
            tf.keras.layers.Embedding(input_dim=V, output_dim=64, name="source_embedding"),
            tf.keras.layers.LSTM(128, name="encoder_lstm"),
            tf.keras.layers.RepeatVector(T, name="repeat_vector"),
            tf.keras.layers.LSTM(128, return_sequences=True, name="decoder_lstm"),
            tf.keras.layers.TimeDistributed(
                tf.keras.layers.Dense(V, activation="softmax"),
                name="token_probabilities",
            ),
        ],
        name="sequence_reversal_encoder_decoder",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


s2s_model = build_seq2seq_model()
s2s_model.summary()

In [ ]:
s2s_checkpoint = OUTPUT_DIR / "exp08_seq2seq_best.keras"
s2s_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(s2s_checkpoint),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

start_time = time.perf_counter()
s2s_history = s2s_model.fit(
    src_train,
    tgt_train,
    epochs=30,
    batch_size=256,
    validation_split=0.1,
    verbose=2,
    callbacks=s2s_callbacks,
)
s2s_seconds = time.perf_counter() - start_time

s2s_test_loss, s2s_test_accuracy = s2s_model.evaluate(src_test, tgt_test, verbose=0)
print(f"Seq2seq training time : {s2s_seconds:.2f} seconds")
print(f"Seq2seq test loss     : {s2s_test_loss:.4f}")
print(f"Seq2seq token accuracy: {s2s_test_accuracy:.4f}")

In [ ]:
s2s_probabilities = s2s_model.predict(src_test, verbose=0)
s2s_predictions = s2s_probabilities.argmax(axis=-1)

token_accuracy = float(np.mean(s2s_predictions == tgt_test))
exact_match_accuracy = float(np.mean(np.all(s2s_predictions == tgt_test, axis=1)))

print(f"Token accuracy      : {token_accuracy:.4f}")
print(f"Exact-sequence match: {exact_match_accuracy:.4f}")
print()
for index in range(5):
    print(f"source   : {src_test[index].tolist()}")
    print(f"expected : {tgt_test[index].tolist()}")
    print(f"predicted: {s2s_predictions[index].tolist()}")
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(s2s_history.history["loss"], label="Train", linewidth=2)
axes[0].plot(s2s_history.history["val_loss"], label="Validation", linewidth=2)
axes[0].set_title("Seq2seq sparse categorical cross-entropy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(s2s_history.history["accuracy"], label="Train", linewidth=2)
axes[1].plot(s2s_history.history["val_accuracy"], label="Validation", linewidth=2)
axes[1].set_title("Seq2seq token accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.0, 1.05)
axes[1].grid(alpha=0.3)
axes[1].legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp08_seq2seq_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 7. Sequence reversal inference

In [ ]:
def reverse_sequence(sequence):
    sequence = np.asarray(sequence, dtype=np.int32).reshape(1, T)
    probabilities = s2s_model.predict(sequence, verbose=0)[0]
    tokens = probabilities.argmax(axis=-1).tolist()
    return tokens


for sequence in [[1, 2, 3, 4, 5, 6], [11, 10, 9, 8, 7, 6], [4, 4, 1, 11, 2, 9]]:
    print(f"{sequence} -> {reverse_sequence(sequence)} (expected {sequence[::-1]})")

## 8. Save models and metrics

In [ ]:
imdb_model_path = OUTPUT_DIR / "exp08_imdb_lstm.keras"
s2s_model_path = OUTPUT_DIR / "exp08_seq2seq_lstm.keras"
imdb_model.save(imdb_model_path)
s2s_model.save(s2s_model_path)

metrics = pd.DataFrame(
    [
        {
            "Task": "IMDB sentiment",
            "Test loss": float(imdb_test_loss),
            "Test accuracy": float(imdb_test_accuracy),
            "Training seconds": float(imdb_seconds),
        },
        {
            "Task": "Sequence reversal",
            "Test loss": float(s2s_test_loss),
            "Test accuracy": float(s2s_test_accuracy),
            "Training seconds": float(s2s_seconds),
        },
    ]
)
metrics.to_csv(OUTPUT_DIR / "exp08_metrics.csv", index=False)
pd.DataFrame(imdb_history.history).to_csv(OUTPUT_DIR / "exp08_imdb_history.csv", index=False)
pd.DataFrame(s2s_history.history).to_csv(OUTPUT_DIR / "exp08_seq2seq_history.csv", index=False)

print(f"Saved IMDB model    : {imdb_model_path.resolve()}")
print(f"Saved seq2seq model : {s2s_model_path.resolve()}")
print(metrics.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

## 9. Conclusion

The IMDB LSTM learns a useful sentiment classifier from raw token sequences.
The encoder-decoder model solves the reversal task almost perfectly because
the mapping is deterministic and the vocabulary is small. Together they show
the two main uses of recurrent networks: sequence classification and
sequence-to-sequence transduction.

The original lab sheet records about **86% accuracy** on IMDB and about
**99.87% token accuracy** on the reversal task. A rerun may differ slightly
because of GPU nondeterminism and early stopping.

**Artifacts created at runtime (not committed):**

- `outputs/exp08_imdb_lstm.keras`
- `outputs/exp08_seq2seq_lstm.keras`
- `outputs/exp08_imdb_best.keras`
- `outputs/exp08_seq2seq_best.keras`
- `outputs/exp08_metrics.csv`
- `outputs/exp08_imdb_history.csv`
- `outputs/exp08_seq2seq_history.csv`
- `outputs/exp08_imdb_confusion_matrix.png`
- `outputs/exp08_imdb_training_curves.png`
- `outputs/exp08_seq2seq_training_curves.png`

**Execution note:** the notebook was syntax-checked during repository
preparation; the full training runs were not performed locally.